In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# Google ADK

The book's listing (pp. 178-179) builds a fallback chain: `primary_handler` tries a precise lookup, `fallback_handler` steps in only if that failed, and `response_agent` reports whatever was found. As printed it cannot run:

| The book | Here |
|---|---|
| `get_precise_location_info` and `get_general_area_info` are never defined | two mock tools: the precise lookup knows one address and fails on anything else; both record the outcome in session state through `tool_context.state` |
| `model="gemini-2.0-flash-exp"` | retired (`404 NOT_FOUND`); `gemini-flash-latest` |
| no runner, no query | a `Runner` and two queries, one the primary tool resolves and one it cannot |

`SequentialAgent` is kept as the book wrote it, so the cell prints `DeprecationWarning: SequentialAgent is deprecated in favor of Workflow`. Cell 10 is the same chain written with `Workflow`.

One thing the listing gets subtly wrong: the instructions say to look at `state["primary_location_failed"]`, but ADK only injects state into an instruction written as `{primary_location_failed}`. The model never sees that flag. The fallback works anyway because it reads the conversation, where the primary handler has already said the lookup failed. The state the tools write is printed after each run, so you can see that it agrees with what the agents did, but nothing reads it.

The code is split: the next cell holds the tools, the book's three instructions and the run helper, with no model. The Gemini cell and the Ollama cell each build the agents and run them, so either can run on its own. The `Workflow` section below is split the same way.

In [3]:
# Book listing: Chapter 12, Hands-On Code Example (ADK), pp. 178-179.
# This cell holds the tools, the book's instructions and a run helper, and no model. The
# Gemini cell and the Ollama cell below each build the agents and call run_location_queries.
import os

from google.adk.agents import Agent, SequentialAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

# ADK 2.7.1 builds tool declarations with an experimental feature that is on by default,
# and warns once per process that it is. This switches it off; ADK falls back to its
# older function-declaration builder.
os.environ.setdefault("ADK_DISABLE_JSON_SCHEMA_FOR_FUNC_DECL", "1")

# --- The two tools the book uses but never defines (mocks) ---
KNOWN_ADDRESSES = {"1600 amphitheatre parkway, mountain view": "37.4220 N, 122.0841 W"}


def get_precise_location_info(address: str, tool_context: ToolContext) -> dict:
    """Looks up the exact coordinates of a street address."""
    print(f"  [tool] get_precise_location_info({address!r})")
    coords = KNOWN_ADDRESSES.get(address.lower().strip(" ."))
    if coords is None:
        tool_context.state["primary_location_failed"] = True
        return {"status": "error", "message": f"Geocoding service could not resolve '{address}'."}
    tool_context.state["primary_location_failed"] = False
    tool_context.state["location_result"] = f"{address}: {coords}"
    return {"status": "success", "location": coords}


def get_general_area_info(city: str, tool_context: ToolContext) -> dict:
    """Returns general, city-level information about a location."""
    print(f"  [tool] get_general_area_info({city!r})")
    tool_context.state["location_result"] = f"General area: {city} (city level only, no coordinates)"
    return {"status": "success", "area": city}


# --- The book's instructions, shared by the Gemini and Ollama agents ---
# Agent 1: Tries the primary tool. Its focus is narrow and clear.
PRIMARY_INSTRUCTION = """
Your job is to get precise location information.
Use the get_precise_location_info tool with the user's provided address.
"""

# Agent 2: Acts as the fallback handler, checking state to decide its action.
FALLBACK_INSTRUCTION = """
Check if the primary location lookup failed by looking at state["primary_location_failed"].
- If it is True, extract the city from the user's original query and use the get_general_area_info tool.
- If it is False, do nothing.
"""

# Agent 3: Presents the final result from the state.
RESPONSE_INSTRUCTION = """
Review the location information stored in state["location_result"].
Present this information clearly and concisely to the user.
If state["location_result"] does not exist or is empty, apologize that you could not retrieve the location.
"""

# --- Run it: one address the primary tool resolves, one it cannot ---
QUERIES = [
    "Where is 1600 Amphitheatre Parkway, Mountain View?",
    "Where is 42 Nowhere Lane, Springfield?",
]


async def run_location_queries(agent) -> None:
    """Runs each query in a fresh session and prints every agent's reply and the final state."""
    runner = InMemoryRunner(agent=agent, app_name="location_app")
    for query in QUERIES:
        session = await runner.session_service.create_session(app_name="location_app", user_id="user_1")
        print(f"\n>>> {query}")
        message = types.Content(role="user", parts=[types.Part(text=query)])
        async for event in runner.run_async(user_id="user_1", session_id=session.id, new_message=message):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text and part.text.strip():
                        print(f"--- {event.author}: {part.text.strip()}")
        session = await runner.session_service.get_session(
            app_name="location_app", user_id="user_1", session_id=session.id
        )
        print(f"state: {session.state}")

In [4]:
# The book's three agents on Gemini, in the book's SequentialAgent.
from google.genai.types import GenerateContentConfig, AutomaticFunctionCallingConfig

GEMINI_MODEL = "gemini-flash-latest"  # the book's gemini-2.0-flash-exp is retired
# ADK runs its own tool loop and never uses the genai SDK's automatic function calling,
# but leaves it enabled, so the SDK logs a one-time warning. Disabling it removes that.
NO_AFC = GenerateContentConfig(
    automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
)

primary_handler = Agent(
    name="primary_handler",
    model=GEMINI_MODEL,
    instruction=PRIMARY_INSTRUCTION,
    tools=[get_precise_location_info],
    generate_content_config=NO_AFC,
)

fallback_handler = Agent(
    name="fallback_handler",
    model=GEMINI_MODEL,
    instruction=FALLBACK_INSTRUCTION,
    tools=[get_general_area_info],
    generate_content_config=NO_AFC,
)

response_agent = Agent(
    name="response_agent",
    model=GEMINI_MODEL,
    instruction=RESPONSE_INSTRUCTION,
    tools=[], # This agent only reasons over the final state.
    generate_content_config=NO_AFC,
)


# The SequentialAgent ensures the handlers run in a guaranteed order.
robust_location_agent = SequentialAgent(
    name="robust_location_agent",
    sub_agents=[primary_handler, fallback_handler, response_agent]
)

await run_location_queries(robust_location_agent)

/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_34240/2725855586.py:37: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  robust_location_agent = SequentialAgent(



>>> Where is 1600 Amphitheatre Parkway, Mountain View?


  [tool] get_precise_location_info('1600 Amphitheatre Parkway, Mountain View')


--- primary_handler: The address 1600 Amphitheatre Parkway, Mountain View is located at coordinates **37.4220° N, 122.0841° W** (the Googleplex headquarters in Mountain View, California).


--- response_agent: 1600 Amphitheatre Parkway in Mountain View, California is located at coordinates **37.4220° N, 122.0841° W**. This is the location of the Googleplex headquarters.
state: {'primary_location_failed': False, 'location_result': '1600 Amphitheatre Parkway, Mountain View: 37.4220 N, 122.0841 W'}

>>> Where is 42 Nowhere Lane, Springfield?


  [tool] get_precise_location_info('42 Nowhere Lane, Springfield')


--- primary_handler: I was unable to find precise location coordinates for "42 Nowhere Lane, Springfield," as the geocoding service could not resolve that address. Please check the address or provide additional details such as a state or postal code.


  [tool] get_general_area_info('Springfield')


--- fallback_handler: Since the exact address "42 Nowhere Lane" could not be located, here is general information for the city of Springfield.


--- response_agent: I was unable to find the specific address at 42 Nowhere Lane, but general area information was located for **Springfield**. 

If you are looking for a specific Springfield, please provide additional details such as a state, postal code, or nearby landmarks.
state: {'primary_location_failed': True, 'location_result': 'General area: Springfield (city level only, no coordinates)'}


## Google ADK + Ollama

In [5]:
# Same fallback chain, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
#   litellm, from the google-adk[extensions] extra (uv sync, then restart the kernel)
# Needs only the shared cell above, not the Gemini cell, and no Google key.
from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

ollama_location_agent = SequentialAgent(
    name="robust_location_agent",
    sub_agents=[
        Agent(
            name="primary_handler",
            model=LiteLlm(model=OLLAMA_MODEL),
            instruction=PRIMARY_INSTRUCTION,
            tools=[get_precise_location_info],
        ),
        Agent(
            name="fallback_handler",
            model=LiteLlm(model=OLLAMA_MODEL),
            instruction=FALLBACK_INSTRUCTION,
            tools=[get_general_area_info],
        ),
        Agent(
            name="response_agent",
            model=LiteLlm(model=OLLAMA_MODEL),
            instruction=RESPONSE_INSTRUCTION,
        ),
    ],
)

await run_location_queries(ollama_location_agent)


>>> Where is 1600 Amphitheatre Parkway, Mountain View?


/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_34240/3717239349.py:11: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  ollama_location_agent = SequentialAgent(


  [tool] get_precise_location_info('1600 Amphitheatre Parkway, Mountain View')


--- primary_handler: The precise location of 1600 Amphitheatre Parkway in Mountain View is at 37.4220 North, 122.0841 West.


--- fallback_handler: For context:
[fallback_handler] Checking if primary location lookup failed...
[fallback_handler] primary_location_failed is False.

Since `primary_location_failed` is False, there's no need to use the `get_general_area_info` tool. The primary location lookup was successful and the coordinates were provided correctly.


...

  [tool] get_precise_location_info('42 Nowhere Lane, Springfield')


--- primary_handler: I'm sorry, but I couldn't find the exact location for "42 Nowhere Lane, Springfield". Could you please provide more details or check the address?


  [tool] get_general_area_info('Springfield')


--- fallback_handler: It seems like the specific address "42 Nowhere Lane" could not be found. However, I can provide you with general information about Springfield. Would you like to know more about the city?


--- response_agent: It seems like the specific address "42 Nowhere Lane, Springfield" could not be found. However, I can provide you with general information about Springfield. Would you like to know more about the city?
state: {'primary_location_failed': True, 'location_result': 'General area: Springfield (city level only, no coordinates)'}


## Google ADK Workflow

The same fallback chain with the graph-based `Workflow` API, which replaces the deprecated `SequentialAgent`. The translation changes *who decides* to fall back:

| `SequentialAgent` (cells 5, 7) | `Workflow` (cells 10, 12) |
|---|---|
| all three agents always run; `fallback_handler` is an LLM deciding whether to act | `check_primary`, a plain Python routing node, reads `primary_location_failed` and sets `ctx.route` |
| the instruction says `state["primary_location_failed"]`, which the model never sees | the flag the tool wrote is actually read, by code |
| `response_agent` is told to look at `state["location_result"]` | its instruction contains `{location_result?}`, so ADK injects the value (the `?` means empty if missing) |

On success the fallback agent does not run at all. That is the pattern in its plainest form: detecting the failure is code, and only recovering from it uses the model.

Two things to know when reading it:

- Edges pass each node's output to the next node, so `fallback_handler` receives the primary handler's reply ("could not resolve '42 Nowhere Lane, Springfield'") and extracts the city from that, not from the user's original message.
- If the primary handler never calls its tool, the flag is missing, and `check_primary` treats that as a failure.

The next cell holds what the two `Workflow` cells share: the routing node and the two new instructions. The primary handler keeps the book's instruction.

In [6]:
# Shared by the Gemini and Ollama Workflow cells below, and no model. Needs the shared
# cell 4 (tools, PRIMARY_INSTRUCTION, run_location_queries), not the SequentialAgent cells.
from google.adk import Workflow
from google.adk.agents.context import Context

# Runs only when check_primary routes here, so it no longer decides whether to act.
WF_FALLBACK_INSTRUCTION = """
The precise location lookup failed. Extract the city from the input and use the get_general_area_info tool.
"""

# {location_result?} is ADK's state injection; the book's state["location_result"] is not.
WF_RESPONSE_INSTRUCTION = """
Location information: {location_result?}
Present this information clearly and concisely to the user.
If it is empty, apologize that you could not retrieve the location.
"""


def check_primary(node_input: str, ctx: Context) -> str:
    """Routes to the fallback when the primary tool recorded a failure, or never ran."""
    failed = ctx.state.get("primary_location_failed", True)
    ctx.route = "failed" if failed else "ok"
    print(f"  [route] primary_location_failed={failed} -> {ctx.route}")
    return node_input

In [7]:
# Same fallback chain, written with the graph-based Workflow API (google-adk 2.x).
# SequentialAgent above is deprecated in favor of Workflow. Here the decision to fall back
# is a routing node reading the state the tool wrote, instead of an agent reading a
# conversation. ADK nodes belong to one parent, so the agents are declared again.
from google.genai.types import GenerateContentConfig, AutomaticFunctionCallingConfig

GEMINI_MODEL = "gemini-flash-latest"  # the book's gemini-2.0-flash-exp is retired
# ADK runs its own tool loop and never uses the genai SDK's automatic function calling,
# but leaves it enabled, so the SDK logs a one-time warning. Disabling it removes that.
NO_AFC = GenerateContentConfig(
    automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
)

wf_primary_handler = Agent(
    name="primary_handler",
    model=GEMINI_MODEL,
    instruction=PRIMARY_INSTRUCTION,
    tools=[get_precise_location_info],
    generate_content_config=NO_AFC,
)

wf_fallback_handler = Agent(
    name="fallback_handler",
    model=GEMINI_MODEL,
    instruction=WF_FALLBACK_INSTRUCTION,
    tools=[get_general_area_info],
    generate_content_config=NO_AFC,
)

wf_response_agent = Agent(
    name="response_agent",
    model=GEMINI_MODEL,
    instruction=WF_RESPONSE_INSTRUCTION,
    generate_content_config=NO_AFC,
)

robust_location_workflow = Workflow(
    name="robust_location_workflow",
    edges=[
        ("START", wf_primary_handler, check_primary),
        (check_primary, {"failed": wf_fallback_handler, "ok": wf_response_agent}),
        (wf_fallback_handler, wf_response_agent),
    ],
)

await run_location_queries(robust_location_workflow)


>>> Where is 1600 Amphitheatre Parkway, Mountain View?


  [tool] get_precise_location_info('1600 Amphitheatre Parkway, Mountain View')


--- primary_handler: 1600 Amphitheatre Parkway, Mountain View is located at coordinates **37.4220° N, 122.0841° W** (the Googleplex headquarters).
  [route] primary_location_failed=False -> ok


--- response_agent: The coordinates for **1600 Amphitheatre Parkway, Mountain View** are **37.4220° N, 122.0841° W**.
state: {'primary_location_failed': False, 'location_result': '1600 Amphitheatre Parkway, Mountain View: 37.4220 N, 122.0841 W'}

>>> Where is 42 Nowhere Lane, Springfield?


  [tool] get_precise_location_info('42 Nowhere Lane, Springfield')


--- primary_handler: The address **42 Nowhere Lane, Springfield** could not be found or resolved by the geocoding service. Please check the street name, number, or specify the state/postal code and try again.
  [route] primary_location_failed=True -> failed


  [tool] get_general_area_info('Springfield')


--- fallback_handler: Since the specific street address could not be resolved, general information for the city of **Springfield** has been retrieved.


--- response_agent: The current location information is set to the general area of **Springfield** (city level only; no specific coordinates or street addresses are available).
state: {'primary_location_failed': True, 'location_result': 'General area: Springfield (city level only, no coordinates)'}


## Google ADK Workflow + Ollama

In [8]:
# Same Workflow, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
#   litellm, from the google-adk[extensions] extra (uv sync, then restart the kernel)
# Needs the shared cells 4 and 9, not the Gemini cells, and no Google key. The routing
# node is plain Python, so only the three agents change.
from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

ollama_primary = Agent(
    name="primary_handler",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction=PRIMARY_INSTRUCTION,
    tools=[get_precise_location_info],
)

ollama_fallback = Agent(
    name="fallback_handler",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction=WF_FALLBACK_INSTRUCTION,
    tools=[get_general_area_info],
)

ollama_response = Agent(
    name="response_agent",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction=WF_RESPONSE_INSTRUCTION,
)

ollama_location_workflow = Workflow(
    name="robust_location_workflow",
    edges=[
        ("START", ollama_primary, check_primary),
        (check_primary, {"failed": ollama_fallback, "ok": ollama_response}),
        (ollama_fallback, ollama_response),
    ],
)

await run_location_queries(ollama_location_workflow)


>>> Where is 1600 Amphitheatre Parkway, Mountain View?


  [tool] get_precise_location_info('1600 Amphitheatre Parkway, Mountain View')


--- primary_handler: The precise location of 1600 Amphitheatre Parkway in Mountain View is at 37.4220 N, 122.0841 W.
  [route] primary_location_failed=False -> ok


--- response_agent: The location of 1600 Amphitheatre Parkway, Mountain View is at 37.4220 N, 122.0841 W.
state: {'primary_location_failed': False, 'location_result': '1600 Amphitheatre Parkway, Mountain View: 37.4220 N, 122.0841 W'}

>>> Where is 42 Nowhere Lane, Springfield?


  [tool] get_precise_location_info('42 Nowhere Lane, Springfield')


...

--- fallback_handler: It seems like the address provided is missing the city name. Could you please provide the city name so I can look up the general information for you? If you're unsure about the city, let me try extracting the city name from the address you provided.

I'll extract the city name from "42 Nowhere Lane, Springfield" and use that to get the general area information.
  [tool] get_general_area_info('Springfield')


--- fallback_handler: The city extracted from the address is Springfield. I will now fetch the general information for Springfield using the get_general_area_info tool.
  [tool] get_general_area_info('Springfield')


  [tool] get_general_area_info('Springfield')


--- fallback_handler: Here is some general information about Springfield:

- Population: [Population data]
- Major industries: [Industries data]
- Notable landmarks: [Landmarks data]
- Climate: [Climate data]

For more specific details, you can provide me with more questions or requests about Springfield.


--- response_agent: Here is the location information for Springfield:

- Population: [Population data]
- Major industries: [Industries data]
- Notable landmarks: [Landmarks data]
- Climate: [Climate data]

For more specific details, feel free to ask!
state: {'primary_location_failed': True, 'location_result': 'General area: Springfield (city level only, no coordinates)'}
